# MHE (Multiple Hash Embeddings) — Criteo Benchmark
**Paper:** "Revisiting the Performance of iALS on Item Recommendation Benchmarks" / 
Hash Embeddings variants (2021)

## Architecture Overview
- **MHE**: Dùng T bảng embedding nhỏ, mỗi category được hash vào mỗi bảng → lấy T vectors → weighted sum → embedding 64 chiều
- **DeepFM & DCN**: 13 integer features → đã `log1p` sẵn → concat trực tiếp; 26 cat features → MHE → 64-dim
- **DLRM**: 13 integer features → MLP → 64-dim; 26 cat features → MHE → 64-dim
- **Metric**: AUC-ROC

In [ ]:
# ============================================================
# Cell 1 — Imports
# ============================================================
import os, math, time, warnings, gc
warnings.filterwarnings('ignore')

try:
    import psutil
except ImportError:
    psutil = None

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

print(f'PyTorch: {torch.__version__}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

In [ ]:
# ============================================================
# Cell 2 — CONFIG
# ============================================================
CFG = {
    # --- Data ---
    'data_dir'  : '/kaggle/input/datasets/nmpogg/criteo-cleaned-v2',

    # --- MHE params ---
    'embed_dim'      : 64,        
    'mhe_num_tables' : 4,         
    'mhe_bucket_size': 1000000,  

    # --- Integer features (cho DLRM) ---
    'int_mlp_dims': [64, 64],

    # --- Training ---
    'batch_size'  : 4096,
    'epochs'      : 5,
    'lr'          : 1e-3,
    'weight_decay': 1e-5,

    # --- Profiling / Resource metrics ---
    'profile_params': True,
    'profile_flops' : True,
    'measure_resources': True,
    'latency_warmup_batches': 5,

    # --- Models ---
    'deepfm_hidden'    : [400, 400, 400],
    'dcn_cross_layers' : 3,
    'dcn_hidden'       : [512, 512],
    'dlrm_top_mlp'     : [512, 256, 128],
}

DENSE_COLS = [f'I{i}' for i in range(1, 14)]
CAT_COLS   = [f'C{i}' for i in range(1, 27)]
LABEL_COL  = 'label'
print('Config loaded.')

In [ ]:
# ============================================================
# Cell 3 — Load Data
# I1-I13 : đã log1p sẵn, dùng trực tiếp
# C1-C26 : string (vd: 'ad3062eb', 'unknownc1'), cần LabelEncoder
# ============================================================
data_dir = CFG['data_dir']

print('Loading parquet splits...')
train_df = pd.read_parquet(os.path.join(data_dir, 'train.parquet'))
val_df   = pd.read_parquet(os.path.join(data_dir, 'validation.parquet'))
test_df  = pd.read_parquet(os.path.join(data_dir, 'test.parquet'))

print(f'Train:      {len(train_df):>10,} rows')
print(f'Validation: {len(val_df):>10,} rows')
print(f'Test:       {len(test_df):>10,} rows')
print(train_df.head(2))

In [ ]:
# ============================================================
# Cell 4 — LabelEncode categorical + cast dtypes
# ============================================================
vocab_sizes = []

for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(train_df[col].astype(str))

    train_df[col] = le.transform(train_df[col].astype(str))

    def safe_transform(series, le=le):
        s = series.astype(str)
        known = set(le.classes_)
        s = s.where(s.isin(known), other=le.classes_[0])
        return le.transform(s)

    val_df[col]  = safe_transform(val_df[col])
    test_df[col] = safe_transform(test_df[col])

    vocab_sizes.append(len(le.classes_))

# Cast dtypes
for df in [train_df, val_df, test_df]:
    df[DENSE_COLS] = df[DENSE_COLS].astype(np.float32)
    df[CAT_COLS]   = df[CAT_COLS].astype(np.int64)
    df[LABEL_COL]  = df[LABEL_COL].astype(np.float32)

# MHE bucket size: min(vocab, mhe_bucket_size) per feature
mhe_buckets = [min(v, CFG['mhe_bucket_size']) for v in vocab_sizes]

print('Vocab sizes (first 5):', vocab_sizes[:5])
print('MHE buckets (first 5):', mhe_buckets[:5])
print('Total cat features   :', len(vocab_sizes))
print('\nDense sample (đã log1p):')
print(train_df[DENSE_COLS].head(2))
print('\nCat encoded:')
print(train_df[CAT_COLS].head(2))

In [ ]:
# ============================================================
# Cell 5 — Train/Val/Test Split
# Dataset đã split sẵn từ `nmpogg/criteo-cleaned-v2`
# ============================================================
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

In [ ]:
# ============================================================
# Cell 6 — Dataset & DataLoader
# ============================================================
class CriteoDataset(Dataset):
    def __init__(self, df, dense_cols, cat_cols, label_col):
        self.dense = torch.tensor(df[dense_cols].values, dtype=torch.float32)
        self.cat   = torch.tensor(df[cat_cols].values,   dtype=torch.long)
        self.label = torch.tensor(df[label_col].values,  dtype=torch.float32)

    def __len__(self): return len(self.label)
    def __getitem__(self, idx): return self.dense[idx], self.cat[idx], self.label[idx]

train_ds = CriteoDataset(train_df, DENSE_COLS, CAT_COLS, LABEL_COL)
val_ds   = CriteoDataset(val_df,   DENSE_COLS, CAT_COLS, LABEL_COL)
test_ds  = CriteoDataset(test_df,  DENSE_COLS, CAT_COLS, LABEL_COL)

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'],   shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size']*2, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=CFG['batch_size']*2, shuffle=False, num_workers=2, pin_memory=True)
print('DataLoaders ready.')

In [ ]:
# ============================================================
# Cell 7 — MHE Module
# ============================================================

class MHEEncoder(nn.Module):
    """
    Multiple Hash Embedding cho một categorical feature.
    Input : (B,) LongTensor
    Output: (B, embed_dim) FloatTensor
    """
    def __init__(self, vocab_size: int, embed_dim: int,
                 num_tables: int = 4, bucket_size: int = 1_000_000):
        super().__init__()
        self.num_tables  = num_tables
        self.bucket_size = min(bucket_size, vocab_size)  
        self.embed_dim   = embed_dim

        self.sub_dim = embed_dim

        self.tables = nn.ModuleList([
            nn.Embedding(self.bucket_size, self.sub_dim)
            for _ in range(num_tables)
        ])

        self.table_weights = nn.Parameter(torch.ones(num_tables) / num_tables)

        self.norm = nn.LayerNorm(embed_dim)

        torch.manual_seed(1337)
        self.register_buffer('a_vec', torch.randint(1, 2**31 - 1, (num_tables,), dtype=torch.long))
        self.register_buffer('b_vec', torch.randint(0, 2**31 - 1, (num_tables,), dtype=torch.long))
        self.prime = 2**31 - 1

    def _hash(self, ids: torch.Tensor, t: int) -> torch.Tensor:
        """Hash ids vào bucket [0, bucket_size) dùng universal hashing."""
        return ((self.a_vec[t] * ids + self.b_vec[t]) % self.prime) % self.bucket_size

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        # ids: (B,)
        weights = F.softmax(self.table_weights, dim=0)  
        emb = torch.zeros(ids.size(0), self.embed_dim, device=ids.device)
        for t in range(self.num_tables):
            hashed = self._hash(ids, t)           
            e = self.tables[t](hashed)             
            emb = emb + weights[t] * e
        return self.norm(emb)  


class MHELayer(nn.Module):
    """
    Wrapper áp dụng MHE cho tất cả 26 cat features.
    Input : (B, 26) LongTensor
    Output: (B, 26, embed_dim)
    """
    def __init__(self, vocab_sizes, embed_dim, num_tables, bucket_size):
        super().__init__()
        self.encoders = nn.ModuleList([
            MHEEncoder(v, embed_dim, num_tables, bucket_size)
            for v in vocab_sizes
        ])
        self.embed_dim = embed_dim

        total_params = sum(sum(p.numel() for p in enc.parameters()) for enc in self.encoders)
        print(f'  MHELayer: {len(vocab_sizes)} features, '
              f'total embedding params: {total_params:,}')

    def forward(self, cat_x: torch.Tensor) -> torch.Tensor:
        embs = [enc(cat_x[:, i]) for i, enc in enumerate(self.encoders)]
        return torch.stack(embs, dim=1)  

print('MHE module defined.')

In [ ]:
# ============================================================
# Cell 8 — Numeric Feature Processors
# ============================================================
class DenseNormalizer(nn.Module):
    """
    Data đã log1p sẵn từ preprocessing.
    Chỉ cần LayerNorm để re-scale trước khi vào model.
    """
    def __init__(self, num_features):
        super().__init__()
        self.norm = nn.LayerNorm(num_features)

    def forward(self, x):
        return self.norm(x)


class DenseEmbedMLP(nn.Module):
    """
    Chỉ dùng cho DLRM: embed mỗi scalar dense feature → vector 64-dim.
    Input : (B, 13)
    Output: (B, 13, embed_dim)
    """
    def __init__(self, num_dense, embed_dim, hidden_dims):
        super().__init__()
        layers = []
        in_d = 1
        for h in hidden_dims:
            layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.GELU()]
            in_d = h
        layers.append(nn.Linear(in_d, embed_dim))
        self.mlp = nn.Sequential(*layers)
        self.norm = nn.LayerNorm(1)
        self.num_dense = num_dense

    def forward(self, x):
        out = []
        for i in range(self.num_dense):
            xi = x[:, i:i+1]       # (B, 1)
            xi = self.norm(xi)
            out.append(self.mlp(xi))
        return torch.stack(out, dim=1)  # (B, 13, embed_dim)

print('Numeric processors defined.')

In [ ]:
# ============================================================
# Cell 9 — DeepFM with MHE  (FIXED: logits, BCEWithLogitsLoss)
# ============================================================
class DeepFM_MHE(nn.Module):
    """
    DeepFM với MHE embedding.
    FIX: output raw logits, dùng BCEWithLogitsLoss.
    FM terms được normalize để tránh gradient explosion.
    """
    def __init__(self, num_dense, vocab_sizes, embed_dim, hidden_dims,
                 mhe_num_tables, mhe_bucket_size):
        super().__init__()
        self.num_dense  = num_dense
        self.num_sparse = len(vocab_sizes)

        self.dense_norm = DenseNormalizer(num_dense)
        self.mhe = MHELayer(vocab_sizes, embed_dim, mhe_num_tables,
                            mhe_bucket_size)  # MHE tự điều chỉnh per feature

        # FM order-1
        self.fm1_sparse_proj = nn.Linear(embed_dim, 1, bias=False)
        self.fm1_dense_proj  = nn.Linear(num_dense, 1)

        # FM order-2 norm
        self.fm2_norm = nn.LayerNorm(1)

        # Deep
        deep_in = num_dense + self.num_sparse * embed_dim
        layers = []
        in_d = deep_in
        for dim in hidden_dims:
            layers += [nn.Linear(in_d, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.1)]
            in_d = dim
        layers.append(nn.Linear(in_d, 1))
        self.deep = nn.Sequential(*layers)

    def forward(self, dense_x, sparse_x):
        dense_norm = self.dense_norm(dense_x)  # (B, 13)
        emb = self.mhe(sparse_x)               # (B, 26, 64)

        # FM-1
        fm1 = self.fm1_sparse_proj(emb).squeeze(-1).sum(1, keepdim=True) + \
              self.fm1_dense_proj(dense_norm)  # (B,1)

        # FM-2 (sparse only)
        sum_emb = emb.sum(1)
        fm2 = 0.5 * (sum_emb**2 - (emb**2).sum(1)).sum(1, keepdim=True)  # (B,1)
        fm2 = self.fm2_norm(fm2)

        # Deep
        deep_in = torch.cat([dense_norm, emb.view(dense_x.size(0), -1)], dim=1)
        deep_out = self.deep(deep_in)

        return fm1 + fm2 + deep_out  # raw logit

print('DeepFM_MHE defined.')

In [ ]:
# ============================================================
# Cell 10 — DCN with MHE  (FIXED)
# ============================================================
class CrossNetwork(nn.Module):
    def __init__(self, input_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        self.w = nn.ParameterList([nn.Parameter(torch.empty(input_dim)) for _ in range(num_layers)])
        self.b = nn.ParameterList([nn.Parameter(torch.zeros(input_dim)) for _ in range(num_layers)])
        for w in self.w:
            nn.init.xavier_uniform_(w.unsqueeze(0))

    def forward(self, x0):
        xl = x0
        for i in range(self.num_layers):
            xlw = (xl * self.w[i]).sum(1, keepdim=True)
            xl  = x0 * xlw + self.b[i] + xl
        return xl


class DCN_MHE(nn.Module):
    """
    DCN với MHE embedding.
    FIX: bỏ sigmoid trong model.
    """
    def __init__(self, num_dense, vocab_sizes, embed_dim, cross_layers, hidden_dims,
                 mhe_num_tables, mhe_bucket_size):
        super().__init__()
        self.num_dense  = num_dense
        self.num_sparse = len(vocab_sizes)
        self.embed_dim  = embed_dim

        self.dense_norm = DenseNormalizer(num_dense)
        self.mhe = MHELayer(vocab_sizes, embed_dim, mhe_num_tables, mhe_bucket_size)

        input_dim = num_dense + self.num_sparse * embed_dim  # 13 + 26*64 = 1677
        self.cross_net = CrossNetwork(input_dim, cross_layers)

        deep_layers = []
        in_d = input_dim
        for dim in hidden_dims:
            deep_layers += [nn.Linear(in_d, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.1)]
            in_d = dim
        self.deep_net = nn.Sequential(*deep_layers)
        self.fc = nn.Linear(input_dim + in_d, 1)

    def forward(self, dense_x, sparse_x):
        dense_norm = self.dense_norm(dense_x)
        emb = self.mhe(sparse_x)                          # (B, 26, 64)
        emb_flat = emb.view(dense_x.size(0), -1)          # (B, 1664)
        x0 = torch.cat([dense_norm, emb_flat], dim=1)     # (B, 1677)

        cross_out = self.cross_net(x0)
        deep_out  = self.deep_net(x0)
        combined  = torch.cat([cross_out, deep_out], dim=1)
        return self.fc(combined)  # raw logit

print('DCN_MHE defined.')

In [ ]:
# ============================================================
# Cell 11 — DLRM with MHE
# ============================================================
class DLRM_MHE(nn.Module):
    """
    DLRM với MHE cho sparse features, MLP embed cho dense features.
    """
    def __init__(self, num_dense, vocab_sizes, embed_dim,
                 dense_embed_hidden, top_mlp_dims,
                 mhe_num_tables, mhe_bucket_size):
        super().__init__()
        self.num_dense  = num_dense
        self.num_sparse = len(vocab_sizes)
        self.embed_dim  = embed_dim

        self.dense_embed = DenseEmbedMLP(num_dense, embed_dim, dense_embed_hidden)
        self.mhe = MHELayer(vocab_sizes, embed_dim, mhe_num_tables, mhe_bucket_size)

        num_all = num_dense + self.num_sparse          # 39
        num_interactions = (num_all * (num_all - 1)) // 2
        top_in = embed_dim + num_interactions

        top_layers = []
        in_d = top_in
        for dim in top_mlp_dims:
            top_layers += [nn.Linear(in_d, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.1)]
            in_d = dim
        top_layers.append(nn.Linear(in_d, 1))
        self.top_mlp = nn.Sequential(*top_layers)

    def forward(self, dense_x, sparse_x):
        dense_emb  = self.dense_embed(dense_x)   # (B, 13, 64)
        sparse_emb = self.mhe(sparse_x)           # (B, 26, 64)
        all_emb    = torch.cat([dense_emb, sparse_emb], dim=1)  # (B, 39, 64)

        Z = torch.bmm(all_emb, all_emb.transpose(1, 2))  # (B, 39, 39)
        n = all_emb.size(1)
        rows, cols = torch.triu_indices(n, n, offset=1)
        interactions = Z[:, rows, cols]  # (B, num_interactions)

        dense_rep = dense_emb.mean(dim=1)  # (B, 64)
        concat = torch.cat([dense_rep, interactions], dim=1)
        return self.top_mlp(concat)  # raw logit

print('DLRM_MHE defined.')

In [ ]:
# ============================================================
# Cell 12 — Profiling Utilities: Params / FLOPs / RAM / VRAM / Latency
# ============================================================
def _safe_gb(value):
    return value / (1024 ** 3)


def current_ram_gb():
    if psutil is None:
        return float('nan')
    return _safe_gb(psutil.virtual_memory().used)


def current_vram_gb(device):
    if device.type != 'cuda':
        return 0.0
    return _safe_gb(torch.cuda.memory_allocated(device))


def peak_vram_gb(device):
    if device.type != 'cuda':
        return 0.0
    return _safe_gb(torch.cuda.max_memory_allocated(device))


def print_parameter_report(model, top_k=40):
    rows = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        rows.append({
            'name': name,
            'shape': tuple(p.shape),
            'params': p.numel(),
            'memory_mb': p.numel() * p.element_size() / (1024 ** 2),
        })

    total_params = sum(r['params'] for r in rows)
    total_param_bytes = sum(p.numel() * p.element_size() for p in model.parameters() if p.requires_grad)
    grad_bytes = total_param_bytes
    adam_state_bytes = total_param_bytes * 2  # Adam/AdamW stores m and v states.
    train_state_bytes = total_param_bytes + grad_bytes + adam_state_bytes

    print(f'Trainable params: {total_params:,}')
    print(f'Parameter memory: {_safe_gb(total_param_bytes):.3f} GB')
    print(f'Grad memory     : {_safe_gb(grad_bytes):.3f} GB')
    print(f'Adam states est.: {_safe_gb(adam_state_bytes):.3f} GB')
    print(f'Train-state est.: {_safe_gb(train_state_bytes):.3f} GB (params + grads + Adam states, no activations)')

    by_root = {}
    for r in rows:
        root = r['name'].split('.')[0]
        by_root[root] = by_root.get(root, 0) + r['params']

    print('\nParams by top module:')
    for root, count in sorted(by_root.items(), key=lambda x: x[1], reverse=True):
        print(f'  {root:<18} {count:>15,}')

    print(f'\nLargest parameter tensors (top {min(top_k, len(rows))}):')
    for r in sorted(rows, key=lambda x: x['params'], reverse=True)[:top_k]:
        print(f"  {r['name']:<55} {str(r['shape']):<22} {r['params']:>12,}  {r['memory_mb']:>9.2f} MB")

    return {
        'trainable_params': total_params,
        'parameter_gb': _safe_gb(total_param_bytes),
        'grad_gb': _safe_gb(grad_bytes),
        'adam_state_gb': _safe_gb(adam_state_bytes),
        'train_state_est_gb': _safe_gb(train_state_bytes),
    }


def profile_model_flops(model, device, num_dense, num_sparse, batch_size=1):
    try:
        from torch.utils.flop_counter import FlopCounterMode
    except Exception as exc:
        print(f'FLOP counter unavailable: {exc}')
        return None

    was_training = model.training
    model.eval()
    dummy_dense = torch.randn(batch_size, num_dense, device=device)
    dummy_cat = torch.zeros(batch_size, num_sparse, dtype=torch.long, device=device)

    print(f'\nFLOPs profile with dummy batch_size={batch_size}:')
    try:
        try:
            flop_counter = FlopCounterMode(model, display=True)
        except TypeError:
            flop_counter = FlopCounterMode(display=True)
        with torch.no_grad(), flop_counter:
            _ = model(dummy_dense, dummy_cat)
        if device.type == 'cuda':
            torch.cuda.synchronize()
    except Exception as exc:
        print(f'FLOP profiling failed: {exc}')
        return None
    finally:
        model.train(was_training)
        del dummy_dense, dummy_cat
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    return True


def profile_model_once(model_name, model, cfg, device, num_dense, num_sparse):
    print('\n' + '-' * 60)
    print(f'Profiling {model_name}-MHE')
    print('-' * 60)

    param_metrics = None
    if cfg.get('profile_params', True):
        param_metrics = print_parameter_report(model)

    if cfg.get('profile_flops', True):
        profile_model_flops(model, device, num_dense, num_sparse, batch_size=1)

    return param_metrics


def summarize_resource_samples(ram_samples, vram_samples, device):
    if len(ram_samples) == 0:
        avg_ram = float('nan')
    else:
        avg_ram = float(np.mean(ram_samples))

    avg_vram = float(np.mean(vram_samples)) if len(vram_samples) else 0.0
    return {
        'avg_ram_gb': avg_ram,
        'avg_vram_gb': avg_vram,
        'peak_vram_gb': peak_vram_gb(device),
    }


In [ ]:
# ============================================================
# Cell 13 — Training Engine
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None, measure_resources=True):
    model.train()
    total_loss = 0
    ram_samples, vram_samples = [], []

    for dense, cat, label in loader:
        dense, cat, label = dense.to(device), cat.to(device), label.to(device)
        optimizer.zero_grad()
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logit = model(dense, cat).squeeze(1)
                loss  = criterion(logit, label)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logit = model(dense, cat).squeeze(1)
            loss  = criterion(logit, label)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item() * label.size(0)

        if measure_resources:
            ram_samples.append(current_ram_gb())
            vram_samples.append(current_vram_gb(device))

    resource_metrics = summarize_resource_samples(ram_samples, vram_samples, device)
    return total_loss / len(loader.dataset), resource_metrics


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    for dense, cat, label in loader:
        dense, cat, label = dense.to(device), cat.to(device), label.to(device)
        logit = model(dense, cat).squeeze(1)
        loss  = criterion(logit, label)
        total_loss += loss.item() * label.size(0)
        all_preds.append(torch.sigmoid(logit).cpu().numpy())
        all_labels.append(label.cpu().numpy())
    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    return total_loss / len(loader.dataset), roc_auc_score(labels, preds)


@torch.no_grad()
def evaluate_with_latency(model, loader, criterion, device, warmup_batches=5, measure_resources=True):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    total_latency_ms, latency_batches, latency_samples = 0.0, 0, 0
    ram_samples, vram_samples = [], []

    use_cuda_timer = device.type == 'cuda'

    for i, (dense, cat, label) in enumerate(loader):
        dense, cat, label = dense.to(device), cat.to(device), label.to(device)
        batch_size = label.size(0)

        if use_cuda_timer:
            start_event = torch.cuda.Event(enable_timing=True)
            end_event = torch.cuda.Event(enable_timing=True)
            start_event.record()
            logit = model(dense, cat).squeeze(1)
            end_event.record()
            torch.cuda.synchronize()
            batch_latency_ms = start_event.elapsed_time(end_event)
        else:
            t0 = time.perf_counter()
            logit = model(dense, cat).squeeze(1)
            batch_latency_ms = (time.perf_counter() - t0) * 1000

        loss  = criterion(logit, label)
        total_loss += loss.item() * batch_size
        all_preds.append(torch.sigmoid(logit).cpu().numpy())
        all_labels.append(label.cpu().numpy())

        if i >= warmup_batches:
            total_latency_ms += batch_latency_ms
            latency_batches += 1
            latency_samples += batch_size

        if measure_resources:
            ram_samples.append(current_ram_gb())
            vram_samples.append(current_vram_gb(device))

    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    auc    = roc_auc_score(labels, preds)

    resource_metrics = summarize_resource_samples(ram_samples, vram_samples, device)
    if latency_batches > 0:
        resource_metrics.update({
            'avg_batch_latency_ms': total_latency_ms / latency_batches,
            'avg_sample_latency_ms': total_latency_ms / max(latency_samples, 1),
        })
    else:
        resource_metrics.update({
            'avg_batch_latency_ms': 0.0,
            'avg_sample_latency_ms': 0.0,
        })

    return total_loss / len(loader.dataset), auc, resource_metrics


def run_experiment(model_name, model, cfg, train_dl, val_dl, test_dl, device):
    sep = '='*60
    print(f'\n{sep}')
    print(f'  Model: {model_name}  |  Embedding: MHE')
    nparams = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Params: {nparams:,}')
    print(f'{sep}')

    model = model.to(device)
    profile_metrics = profile_model_once(
        model_name, model, cfg, device,
        num_dense=len(DENSE_COLS),
        num_sparse=len(CAT_COLS),
    )

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg['lr'], steps_per_epoch=len(train_dl), epochs=cfg['epochs']
    )
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    best_val_auc = 0
    results = []
    measure_resources = cfg.get('measure_resources', True)

    for epoch in range(1, cfg['epochs'] + 1):
        if device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats(device)

        t0 = time.time()
        train_loss, train_resources = train_one_epoch(
            model, train_dl, optimizer, criterion, device, scaler,
            measure_resources=measure_resources,
        )
        scheduler.step()
        val_loss, val_auc = evaluate(model, val_dl, criterion, device)
        elapsed = time.time() - t0

        print(
            f'  Epoch {epoch}/{cfg["epochs"]} | Train Loss: {train_loss:.4f} | '
            f'Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | {elapsed:.1f}s'
        )
        print(
            f'    Resources | RAM avg: {train_resources["avg_ram_gb"]:.2f} GB | '
            f'VRAM avg: {train_resources["avg_vram_gb"]:.2f} GB | '
            f'VRAM peak: {train_resources["peak_vram_gb"]:.2f} GB'
        )

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_auc': val_auc,
            'epoch_seconds': elapsed,
            **train_resources,
        }
        if profile_metrics is not None:
            row.update(profile_metrics)
        results.append(row)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), f'{model_name}_MHE_best.pt')

    model.load_state_dict(torch.load(f'{model_name}_MHE_best.pt', map_location=device))
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)

    test_loss, test_auc, test_metrics = evaluate_with_latency(
        model, test_dl, criterion, device,
        warmup_batches=cfg.get('latency_warmup_batches', 5),
        measure_resources=measure_resources,
    )
    print(f'\n   Test AUC: {test_auc:.4f}  |  Test Loss: {test_loss:.4f}')
    print(
        f'   Test resources | RAM avg: {test_metrics["avg_ram_gb"]:.2f} GB | '
        f'VRAM avg: {test_metrics["avg_vram_gb"]:.2f} GB | '
        f'VRAM peak: {test_metrics["peak_vram_gb"]:.2f} GB'
    )
    print(
        f'   Latency | Batch: {test_metrics["avg_batch_latency_ms"]:.2f} ms | '
        f'Sample: {test_metrics["avg_sample_latency_ms"]:.6f} ms'
    )

    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    return results, test_auc

print('Training engine ready.')

In [ ]:
# ============================================================
# Cell 14 — Memory Analysis: MHE vs Standard Embedding
# ============================================================
print('MHE vs Standard Embedding — Memory Analysis')
print('-' * 60)
embed_dim   = CFG['embed_dim']
num_tables  = CFG['mhe_num_tables']
bucket_size = CFG['mhe_bucket_size']

total_std, total_mhe = 0, 0
for i, (col, vs) in enumerate(zip(CAT_COLS, vocab_sizes)):
    std_params = vs * embed_dim
    b = min(bucket_size, vs)
    mhe_params = num_tables * b * embed_dim
    ratio = mhe_params / std_params if std_params > 0 else 1
    total_std += std_params
    total_mhe += mhe_params
    if i < 5 or vs > 100000:
        print(f'  {col}: vocab={vs:,}, std={std_params:,}, mhe={mhe_params:,}, ratio={ratio:.2f}x')

print(f'\n  TOTAL  std={total_std:,}, mhe={total_mhe:,}, ratio={total_mhe/total_std:.2f}x')
print(f'  Savings: {(1 - total_mhe/total_std)*100:.1f}%' if total_mhe < total_std else '  MHE uses more memory for small vocabs — bucket_size auto-adjusted.')

In [ ]:
# ============================================================
# Cell 15 — Run DeepFM + MHE
# ============================================================
deepfm_model = DeepFM_MHE(
    num_dense       = len(DENSE_COLS),
    vocab_sizes     = vocab_sizes,
    embed_dim       = CFG['embed_dim'],
    hidden_dims     = CFG['deepfm_hidden'],
    mhe_num_tables  = CFG['mhe_num_tables'],
    mhe_bucket_size = CFG['mhe_bucket_size'],
)
deepfm_results, deepfm_test_auc = run_experiment(
    'DeepFM', deepfm_model, CFG, train_dl, val_dl, test_dl, DEVICE
)

In [ ]:
# ============================================================
# Cell 16 — Run DCN + MHE
# ============================================================
dcn_model = DCN_MHE(
    num_dense      = len(DENSE_COLS),
    vocab_sizes    = vocab_sizes,
    embed_dim      = CFG['embed_dim'],
    cross_layers   = CFG['dcn_cross_layers'],
    hidden_dims    = CFG['dcn_hidden'],
    mhe_num_tables = CFG['mhe_num_tables'],
    mhe_bucket_size= CFG['mhe_bucket_size'],
)
dcn_results, dcn_test_auc = run_experiment(
    'DCN', dcn_model, CFG, train_dl, val_dl, test_dl, DEVICE
)

In [ ]:
# ============================================================
# Cell 17 — Run DLRM + MHE
# ============================================================
dlrm_model = DLRM_MHE(
    num_dense          = len(DENSE_COLS),
    vocab_sizes        = vocab_sizes,
    embed_dim          = CFG['embed_dim'],
    dense_embed_hidden = CFG['int_mlp_dims'],
    top_mlp_dims       = CFG['dlrm_top_mlp'],
    mhe_num_tables     = CFG['mhe_num_tables'],
    mhe_bucket_size    = CFG['mhe_bucket_size'],
)
dlrm_results, dlrm_test_auc = run_experiment(
    'DLRM', dlrm_model, CFG, train_dl, val_dl, test_dl, DEVICE
)

In [ ]:
# ============================================================
# Cell 18 — Summary & Comparison
# ============================================================
print('\n' + '='*60)
print('  FINAL RESULTS — MHE Embedding')
print('='*60)
print(f'  DeepFM-MHE  Test AUC: {deepfm_test_auc:.4f}')
print(f'  DCN-MHE     Test AUC: {dcn_test_auc:.4f}')
print(f'  DLRM-MHE    Test AUC: {dlrm_test_auc:.4f}')
print('='*60)

summary = pd.DataFrame({
    'model'     : ['DeepFM-MHE', 'DCN-MHE', 'DLRM-MHE'],
    'embedding' : ['MHE'] * 3,
    'dataset'   : ['nmpogg/criteo-cleaned-v2'] * 3,
    'test_auc'  : [deepfm_test_auc, dcn_test_auc, dlrm_test_auc],
})
summary